In [1]:
"""
Created on Mon Aug  4 16:07:26 2025

@author: huzefa
"""

'\nCreated on Mon Aug  4 16:07:26 2025\n\n@author: huzefa\n'

In [2]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import confusion_matrix, accuracy_score, recall_score, average_precision_score, make_scorer
from sklearn.neural_network import MLPClassifier

In [3]:
#Importing dataset
diabetes=pd.read_excel('../Datasets/data_file.xlsx')
diabetes.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2703 entries, 0 to 2702
Data columns (total 9 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   GENDER    2703 non-null   int64  
 1   AGE       2703 non-null   int64  
 2   Height    2703 non-null   int64  
 3   Weight    2703 non-null   float64
 4   BMI       2703 non-null   float64
 5   BAI       2703 non-null   float64
 6   HBA1C1    2703 non-null   float64
 7   OGTT1FBS  2703 non-null   int64  
 8   NDD       2703 non-null   int64  
dtypes: float64(4), int64(5)
memory usage: 190.2 KB


In [4]:
#Creating a copy of original data to work upon
clean_data=diabetes.copy()

In [5]:
#Pruning duplicates from copy data
clean_data=clean_data.drop_duplicates(keep='first')
print(clean_data.shape)
print(clean_data.head())

(1065, 9)
   GENDER  AGE  Height  Weight        BMI    BAI  HBA1C1  OGTT1FBS  NDD
0       1   37     156    88.0  36.160421  41.53     5.1       102    0
1       0   35     146    56.0  26.271345  34.15     5.0        91    0
2       1   54     160    76.0  29.687500  28.45     5.4        73    0
3       0   46     154    64.0  26.986001  33.28     6.0       151    0
4       0   70     156    55.0  22.600263  21.52     5.6       142    0


In [6]:
#Assiging labels with appropriate numerics
nondia=-1
diabetic=1

In [7]:
#Creating a dataframe with a column depicting diabetic status
Ynew=pd.DataFrame(nondia,index=clean_data.index,columns=['Diabetic'])

In [8]:
#Identifying the diabetic status of each record using the blood test results of FBS or HBA1C1
Ynew.iloc[list(np.where((clean_data.OGTT1FBS>=126) | (clean_data.HBA1C1>=6.5))[0])]=diabetic

In [9]:
#Concatenating the diabetic status with the anthroprometric features of the dataset
data_df=pd.concat([clean_data.iloc[:,:6],Ynew],axis=1)
print(data_df.head())

   GENDER  AGE  Height  Weight        BMI    BAI  Diabetic
0       1   37     156    88.0  36.160421  41.53        -1
1       0   35     146    56.0  26.271345  34.15        -1
2       1   54     160    76.0  29.687500  28.45        -1
3       0   46     154    64.0  26.986001  33.28         1
4       0   70     156    55.0  22.600263  21.52         1


In [10]:
#Finding the diabetic and non-diabetic patients
diabetic_yes=data_df.iloc[list(np.where(data_df.Diabetic==diabetic)[0])]
diabetic_no=data_df.iloc[list(np.where(data_df.Diabetic==nondia)[0])]

In [11]:
#Finding basic stats about both classes respectively
print(diabetic_yes.describe())
print(diabetic_no.describe())

           GENDER         AGE      Height      Weight         BMI         BAI  \
count  528.000000  528.000000  528.000000  528.000000  528.000000  528.000000   
mean     0.571970   51.812500  160.280303   68.404356   26.640045   29.618864   
std      0.495262   10.921528    7.562330   12.759664    4.809005    7.753363   
min      0.000000   25.000000  138.000000   36.500000   15.390454    8.300000   
25%      0.000000   43.750000  156.000000   59.000000   23.290154   24.790000   
50%      1.000000   50.000000  159.500000   68.000000   26.527004   28.145000   
75%      1.000000   59.000000  165.000000   76.000000   29.585799   33.847500   
max      1.000000   80.000000  186.000000  102.000000   41.207076   58.250000   

       Diabetic  
count     528.0  
mean        1.0  
std         0.0  
min         1.0  
25%         1.0  
50%         1.0  
75%         1.0  
max         1.0  
           GENDER         AGE      Height      Weight         BMI         BAI  \
count  537.000000  537.0000

In [12]:
#Splitting data into training and testing set
train_x, test_x, train_y, test_y=train_test_split(data_df.iloc[:,:6], data_df.Diabetic,test_size=0.3,random_state=43)

In [13]:
#Normalizing data using standard scalar
sc=StandardScaler()
train_x=sc.fit_transform(train_x)
test_x=sc.transform(test_x)

In [14]:
#Fetching the diabetic records from train set
diabetic_yes_train=train_x[list(np.where(train_y==diabetic)[0])]

In [15]:
#Fetching the non-diabetic records from train set
diabetic_no_train=train_x[list(np.where(train_y==nondia)[0])]

In [16]:
#Displaying the counts for each class
print('non-diabetic=',diabetic_no_train.shape,'diabetic=',diabetic_yes_train.shape)

non-diabetic= (369, 6) diabetic= (376, 6)


In [17]:
#Fetching the diabetic records from test set
diabetic_yes_test=test_x[list(np.where(test_y==diabetic)[0])]

In [18]:
#Fetching the non-diabetic records from test set
diabetic_no_test=test_x[list(np.where(test_y==nondia)[0])]

In [19]:
#Displaying the counts for each class from test set
print('non-diabetic=',diabetic_no_test.shape,'diabetic=',diabetic_yes_test.shape)

non-diabetic= (168, 6) diabetic= (152, 6)


In [20]:
#Functions for evaluating model using confusion matrix and accuracy score between true and actual
def evaluate(yt,yp):
    cf=confusion_matrix(yt,yp)
    acc=accuracy_score(yt,yp)
    return cf,acc

In [21]:
# Display metrics
def display(yt,yp,model):
    cf,acc = evaluate(yt,yp)
    print('Model=',model,'\ncf=',cf,'\n','\nacc=',acc,'\n')

In [22]:
#Performing classification using MLP classifier
mlpc = MLPClassifier(hidden_layer_sizes=(1), activation='tanh', learning_rate='invscaling', max_iter=10000, solver='sgd', random_state=0, early_stopping=True)
mlpc.fit(train_x, train_y)
train_yp=mlpc.predict(train_x)
test_yp=mlpc.predict(test_x)

In [23]:
#Displaying the results
display(train_y,train_yp,'MLP: Training')
display(test_y,test_yp,'MLP: Testing')

Model= MLP: Training 
cf= [[  0 369]
 [  0 376]] 
 
acc= 0.5046979865771812 

Model= MLP: Testing 
cf= [[  0 168]
 [  0 152]] 
 
acc= 0.475 



In [24]:
#Attributes of the MLP Classifier
print(mlpc.classes_)
print(mlpc.loss_)
print(mlpc.coefs_)
print(mlpc.intercepts_)
print(mlpc.n_layers_)
print(mlpc.n_iter_)
print(mlpc.n_outputs_)
print(mlpc.out_activation_)

[-1  1]
0.8712554959856844
[array([[ 0.09068052],
       [ 0.39944042],
       [ 0.1902262 ],
       [ 0.08273035],
       [-0.14168853],
       [ 0.26935191]]), array([[1.35782622]])]
[array([-0.11940647]), array([1.60265222])]
3
12
1
logistic


In [25]:
#Hyperparameter tuning using GridSearchCV
custom_scorer = {'recall':make_scorer(recall_score, pos_label=diabetic), 'precision':make_scorer(average_precision_score, pos_label=diabetic)}

In [26]:
gscv = GridSearchCV(MLPClassifier(max_iter=10000,random_state=0),
{'activation':('tanh','logistic','relu'),
'hidden_layer_sizes':range(1,4,1),'solver':['adam','sgd']},
cv=5,verbose=False,
scoring=custom_scorer,refit='recall')
gscv.fit(train_x,train_y)
gscv.best_params_

{'activation': 'logistic', 'hidden_layer_sizes': 1, 'solver': 'adam'}

In [27]:
#Performoming Classification using MLP Classifier with best obtained parameters
mlpc = MLPClassifier(hidden_layer_sizes=(1),activation='logistic',
max_iter=10000,
solver='adam',
random_state=0)
mlpc.fit(train_x, train_y)
train_yp=mlpc.predict(train_x)
test_yp=mlpc.predict(test_x)

In [28]:
#Displaying the results
display(train_y,train_yp,'MLP with "sgd" solver and 1,4 hidden nodes')
display(test_y,test_yp,'For Testing')

Model= MLP with "sgd" solver and 1,4 hidden nodes 
cf= [[202 167]
 [ 96 280]] 
 
acc= 0.6469798657718121 

Model= For Testing 
cf= [[ 92  76]
 [ 42 110]] 
 
acc= 0.63125 



In [29]:
#Attributes of improved model
print(mlpc.coefs_)
print(mlpc.intercepts_)
print(mlpc.score(test_x,test_y))

[array([[ 0.41181678],
       [ 1.50426546],
       [ 0.54746517],
       [-0.06032402],
       [-0.33931276],
       [-0.38891654]]), array([[0.75813057]])]
[array([-0.55865635]), array([-0.1876706])]
0.63125
